Representación del entorno

In [20]:
import asyncio
import json
import heapq
from dataclasses import dataclass, field
from enum import Enum
from typing import List, Tuple, Dict, Optional
from autogen_core import RoutedAgent, message_handler, SingleThreadedAgentRuntime, AgentId, DefaultTopicId, TypePrefixSubscription
from pydantic import BaseModel

In [21]:
{
  "config": {
    "dimensiones": [10, 10],
    "iteraciones_max": 100,
    "num_depositos": 1,
    "num_clientes": 1,
    "num_vehiculos":1,
  },
  "mapa": [
    [".", ".", "#", "D", "."],
    ["T", ".", ".", ".", "."],
    [".", "#", "C", ".", "T"]
  ],
  "entidades": {
    "deposito_1": {
      "ubicacion": [0, 3],
      "inventario": 100,
      "paquetes": {"pares": 50, "impares": 50}
    },
    "cliente_1": {
      "ubicacion": [2, 2],
      "demanda": 30,
      "prioridad": "alta"
    },
    "vehiculo_1": {
      "ubicacion": [0, 0],
      "capacidad": 20,
      "coste_paso": 1.5
    }
  }
}

{'config': {'dimensiones': [10, 10],
  'iteraciones_max': 100,
  'num_depositos': 1,
  'num_clientes': 1,
  'num_vehiculos': 1},
 'mapa': [['.', '.', '#', 'D', '.'],
  ['T', '.', '.', '.', '.'],
  ['.', '#', 'C', '.', 'T']],
 'entidades': {'deposito_1': {'ubicacion': [0, 3],
   'inventario': 100,
   'paquetes': {'pares': 50, 'impares': 50}},
  'cliente_1': {'ubicacion': [2, 2], 'demanda': 30, 'prioridad': 'alta'},
  'vehiculo_1': {'ubicacion': [0, 0], 'capacidad': 20, 'coste_paso': 1.5}}}

Modelado de Objetos

In [22]:
from dataclasses import dataclass, field
from enum import IntEnum, Enum
from typing import List, Tuple, Dict, Optional
import heapq

In [ ]:
class Prioridad(IntEnum):
    ALTA = 3
    MEDIA = 2
    BAJA = 1

class Celda(str, Enum):
    LIBRE = "."
    OBSTACULO = "#"
    DEPOSITO = "D"
    CLIENTE = "C"
    TRAFICO = "T"

In [24]:
class EntidadLogistica(BaseModel):
    id: str
    posicion: Tuple[int, int]

class Vehiculo(EntidadLogistica):
    capacidad_max: int
    carga_actual: int = 0
    coste_paso: float = 1.0
    inventario_interno: Dict[str, int] = {}
    
class Cliente(EntidadLogistica):
    demandas: Dict[str, int]
    prioridad: Prioridad
    recibido: Dict[str, int] = {} # Inicializado como diccionario vacío

class Deposito(EntidadLogistica):
    capacidad_max: int
    paquetes_pares: int
    paquetes_impares: int
    
    @property
    def inventario_actual(self) -> int:
        return (self.paquetes_pares * 2) + self.paquetes_impares

class EstadoEntorno(BaseModel):
    mapa: List[List[str]]
    depositos: List[Deposito]
    clientes: List[Cliente]
    vehiculos: List[Vehiculo]
    dimensiones: Tuple[int, int]
    penalizacion_trafico : int

class MensajeLog(BaseModel):
    iteracion: int
    agente: str
    accion: str
    detalles: str

Configuración

In [25]:
def cargar_configuracion(ruta_archivo: str) -> EstadoEntorno:
    with open(ruta_archivo, 'r') as f:
        data = json.load(f)
    
    # Conversión explícita a Tuplas para evitar errores en A*
    depositos = [Deposito(id=d['id'], posicion=tuple(d['posicion']), 
                          capacidad_max=d['capacidad_max'],
                          paquetes_pares=d['paquetes_pares'], 
                          paquetes_impares=d['paquetes_impares']) 
                 for d in data['entidades']['depositos']]
                 
    clientes = [Cliente(id=c['id'], posicion=tuple(c['posicion']), 
                        demandas=c['demandas'], 
                        prioridad=Prioridad[c['prioridad']]) 
                for c in data['entidades']['clientes']]
                
    vehiculos = [Vehiculo(id=v['id'], posicion=tuple(v['posicion']), 
                          capacidad_max=v['capacidad_max'], 
                          coste_paso=v['coste_paso']) 
                 for v in data['entidades']['vehiculos']]
    
    return EstadoEntorno(
        mapa=data['mapa'],
        depositos=depositos,
        clientes=clientes,
        vehiculos=vehiculos,
        dimensiones=tuple(data['config']['dimensiones']),
        penalizacion_trafico = data['config']['penalizacion_trafico']
    )

Entorno no coordinado

In [26]:
from autogen_core import RoutedAgent, message_handler
from dataclasses import dataclass

In [27]:
class AgenteVehiculoSimple(RoutedAgent):
    def __init__(self, id, capacidad, coste_paso, penalizacion_t, posicion_inicial=(0,0)):
        super().__init__(id)
        self.capacidad_max = capacidad  # Nombre corregido
        self.coste_paso = coste_paso
        self.posicion = posicion_inicial
        self.carga_actual = 0
        self.plan_ruta = []
        self.penalizacion_t = penalizacion_t
        self.inventario_interno = {}

    @message_handler
    async def actualizar_y_decidir(self, message: EstadoEntorno, ctx) -> None:
        # Percepción y Planificación reactiva [cite: 21, 22]
        if not self.plan_ruta:
            objetivo = self.seleccionar_objetivo_inteligente(message)
            if objetivo:
                self.plan_ruta = self.calcular_a_estrella(self.posicion, objetivo, message.mapa)
        
        # Acción de movimiento [cite: 21]
        if self.plan_ruta:
            self.posicion = self.plan_ruta.pop(0)
            for v_entorno in message.vehiculos:
                if v_entorno.id == self.id.type:
                    v_entorno.posicion = self.posicion
                    v_entorno.carga_actual = self.carga_actual
            print(f"🚚 Vehículo {self.id} se mueve a {self.posicion} (Carga: {self.carga_actual})")
            self.verificar_llegada(message)
        
    def estimar_coste_ruta(self, destino, mapa):
        """
        Calcula el coste real de una ruta considerando distancia y tráfico.
        coste = coste_distancia + penalizacion_trafico.
        """
        ruta_tentativa = self.calcular_a_estrella(self.posicion, destino, mapa)
        if not ruta_tentativa:
            return float('inf')
        
        coste_total = 0
        for paso in ruta_tentativa:
            coste_total += self.coste_paso
            # Si la celda es zona de tráfico (T), se aplica la penalización [cite: 10, 45]
            if mapa[paso[0]][paso[1]] == 'T':
                coste_total += self.penalizacion_t
        return coste_total

    def seleccionar_objetivo_inteligente(self, estado_entorno):
        """
        Selecciona el objetivo basándose en prioridad del cliente y coste de ruta.
        """
        candidatos_clientes = []
        candidatos_depositos = []

        # 1. Filtrar Clientes: Solo aquellos cuya mercancía ya llevamos [cite: 18]
        if self.carga_actual > 0:
            for c in estado_entorno.clientes:
                for dep_id, cant_en_camion in self.inventario_interno.items():
                    if cant_en_camion > 0 and c.demandas.get(dep_id, 0) > c.recibido.get(dep_id, 0):
                        candidatos_clientes.append(c)
                        break

        # 2. Filtrar Depósitos: Solo si hay espacio y demanda pendiente [cite: 14, 15]
        if self.carga_actual < self.capacidad_max:
            for d in estado_entorno.depositos:
                if d.paquetes_pares > 0 or d.paquetes_impares > 0:
                    if any(c.demandas.get(d.id, 0) > c.recibido.get(d.id, 0) for c in estado_entorno.clientes):
                        candidatos_depositos.append(d)

        # 3. CRITERIO DE PRIORIDAD Y COSTE [cite: 31, 33, 34]
        if candidatos_clientes:
            mejor_cliente = min(candidatos_clientes, key=lambda c: (
                -c.prioridad.value, # Orden descendente: Alta(3), Media(2), Baja(1)
                self.estimar_coste_ruta(c.posicion, estado_entorno.mapa) # Menor coste real
            ))
            return mejor_cliente.posicion

        if candidatos_depositos:
            # Si no hay entregas urgentes, buscamos el depósito con menor coste de acceso
            mejor_depo = min(candidatos_depositos, key=lambda d: self.estimar_coste_ruta(d.posicion, estado_entorno.mapa))
            return mejor_depo.posicion
        return None
        
    def calcular_a_estrella(self, inicio, fin, mapa):
        # 8 direcciones de movimiento 
        direcciones = [(0, 1), (0, -1), (1, 0), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1)]
        open_list = [(0, inicio)]
        camino_previo = {}
        coste_acumulado = {inicio: 0}

        while open_list:
            _, actual = heapq.heappop(open_list)
            if actual == fin: return self.reconstruir_camino(camino_previo, actual)
            
            for dx, dy in direcciones:
                vecino = (actual[0] + dx, actual[1] + dy)
                if self.es_transitable(vecino, mapa):
                    # Coste adicional si es zona de tráfico (T) [cite: 10, 45]
                    coste = self.coste_paso + (self.penalizacion_t if mapa[vecino[0]][vecino[1]] == 'T' else 0)
                    nuevo_coste = coste_acumulado[actual] + coste
                    
                    if vecino not in coste_acumulado or nuevo_coste < coste_acumulado[vecino]:
                        coste_acumulado[vecino] = nuevo_coste
                        prioridad = nuevo_coste + self.distancia_chebyshev(vecino, fin)
                        heapq.heappush(open_list, (prioridad, vecino))
                        camino_previo[vecino] = actual
        return []

    def distancia_chebyshev(self, p1, p2):
        return max(abs(p1[0] - p2[0]), abs(p1[1] - p2[1]))

    def es_transitable(self, pos, mapa):
        # Evita obstáculos (#) 
        return 0 <= pos[0] < len(mapa) and 0 <= pos[1] < len(mapa[0]) and mapa[pos[0]][pos[1]] != '#'

    def reconstruir_camino(self, camino_previo, actual):
        camino = []
        while actual in camino_previo:
            camino.append(actual)
            actual = camino_previo[actual]
        return camino[::-1]

    def verificar_llegada(self, estado):
        # CARGA: Lógica de paquetes pares e impares [cite: 14, 15]
        for depo in estado.depositos:
            if self.posicion == depo.posicion and self.carga_actual < self.capacidad_max:
                espacio = self.capacidad_max - self.carga_actual
                pares = min(espacio // 2, depo.paquetes_pares)
                impares = min(espacio - (pares * 2), depo.paquetes_impares)
                unidades = (pares * 2) + impares
            
                if unidades > 0:
                    self.inventario_interno[depo.id] = self.inventario_interno.get(depo.id, 0) + unidades
                    self.carga_actual += unidades
                    depo.paquetes_pares -= pares
                    depo.paquetes_impares -= impares
                    print(f"📦 {self.id} cargó {unidades} de {depo.id}. Total: {self.carga_actual}")
                    self.plan_ruta = []
                return

        # ENTREGA: Basada en la demanda específica por depósito
        for c in estado.clientes:
            if self.posicion == c.posicion:
                entregado_en_esta_parada = False
                for dep_id in list(self.inventario_interno.keys()):
                    cant_en_camion = self.inventario_interno[dep_id]
                    pendiente = c.demandas.get(dep_id, 0) - c.recibido.get(dep_id, 0)
                    
                    entrega = min(cant_en_camion, pendiente)
                    if entrega > 0:
                        c.recibido[dep_id] = c.recibido.get(dep_id, 0) + entrega
                        self.inventario_interno[dep_id] -= entrega
                        self.carga_actual -= entrega
                        entregado_en_esta_parada = True
                        print(f"🚚 {self.id} entregó {entrega} (Origen: {dep_id}) a {c.id}")
                
                if entregado_en_esta_parada:
                    self.plan_ruta = []
                return               

In [28]:
def imprimir_mapa_coloreado(estado: EstadoEntorno):
    RESET = "\033[0m"
    GRIS = "\033[90m"
    ROJO = "\033[91m"
    VERDE = "\033[92m"
    AMARILLO = "\033[93m"
    CYAN = "\033[96m"

    # Copia para no destruir el mapa original
    grid_visual = [fila[:] for fila in estado.mapa]
    
    # Dibujar Vehículos (Verde)
    for v in estado.vehiculos:
        r, c = v.posicion
        grid_visual[r][c] = f"{VERDE}V{RESET}"

    print(f"\n--- ITERACIÓN ---")
    for r in range(len(grid_visual)):
        fila_coloreada = []
        for c in range(len(grid_visual[0])):
            char = grid_visual[r][c]
            # Colorear según el tipo de celda [cite: 8, 9, 10]
            if char == "#": fila_coloreada.append(f"{GRIS}#{RESET}")
            elif char == "D": fila_coloreada.append(f"{ROJO}D{RESET}")
            elif char == "C": fila_coloreada.append(f"{AMARILLO}C{RESET}")
            elif char == "T": fila_coloreada.append(f"{CYAN}T{RESET}")
            else: fila_coloreada.append(char)
        print(" ".join(fila_coloreada))

Prueba

In [29]:
import json

In [30]:
def generar_escenario_prueba():
    escenario = {
        "config": {
            "dimensiones": [10, 10],
            "penalizacion_trafico": 5
        },
        "mapa": [
            [".", ".", ".", ".", ".", "#", ".", ".", ".", "."],
            [".", "D", ".", ".", ".", "#", ".", "T", "T", "."],
            [".", ".", ".", ".", ".", ".", ".", "T", "C", "."],
            [".", "#", "#", "#", ".", ".", ".", ".", ".", "."],
            [".", ".", ".", ".", ".", ".", ".", ".", ".", "."],
            [".", "C", ".", "T", ".", "D", ".", ".", ".", "."],
            [".", ".", ".", "T", ".", ".", ".", "#", "#", "."],
            [".", ".", ".", ".", ".", ".", ".", "#", ".", "."],
            [".", ".", ".", ".", ".", ".", ".", ".", ".", "."],
            [".", ".", ".", ".", ".", ".", ".", ".", ".", "."]
        ],
        "entidades": {
            "depositos": [
                {"id": "D1", "posicion": [1, 1], "paquetes_pares": 10, "paquetes_impares": 5, "capacidad_max": 15},
                {"id": "D2", "posicion": [5, 5], "paquetes_pares": 5, "paquetes_impares": 10, "capacidad_max": 15}
            ],
            "clientes": [
                {
                    "id": "C1", 
                    "posicion": [2, 8], 
                    "demandas": {"D1": 10, "D2": 5}, 
                    "prioridad": "ALTA"
                },
                {
                    "id": "C2", 
                    "posicion": [5, 1], 
                    "demandas": {"D2": 15}, 
                    "prioridad": "MEDIA"
                }
            ],
            "vehiculos": [
                {"id": "V1", "posicion": [8, 1], "capacidad_max": 10, "coste_paso": 1.0}
            ]
        }
    }
    
    with open("escenario_logistica.json", "w") as f:
        json.dump(escenario, f, indent=4)
    print("✅ Escenario JSON generado con éxito.")

generar_escenario_prueba()

✅ Escenario JSON generado con éxito.


In [31]:
class MonitorMetricas:
    def __init__(self):
        self.tiempo_total = 0      # Iteraciones [cite: 50]
        self.desplazamientos = 0   # Suma total de movimientos [cite: 51]
        self.coste_acumulado = 0   # Suma de costes de paso + tráfico [cite: 53]
        self.espacio_desperdiciado = 0 # Capacidad no usada en ruta [cite: 52]
        self.historial_entregas = [] # Para calcular tiempos por prioridad [cite: 54]

    def registrar_paso(self, vehiculo, celda_tipo, penalizacion_t):
        self.desplazamientos += 1
        coste = vehiculo.coste_paso
        if celda_tipo == 'T':
            coste += penalizacion_t
        self.coste_acumulado += coste
        # Espacio desperdiciado en este paso [cite: 52]
        if vehiculo.carga_actual > 0:
            self.espacio_desperdiciado += (vehiculo.capacidad_max - vehiculo.carga_actual)

    def mostrar_resumen(self):
        print("\n" + "="*30)
        print("📊 MÉTRICAS DE LA SIMULACIÓN")
        print("="*30)
        print(f"⏱️ Tiempo total (it): {self.tiempo_total}") 
        print(f"🛣️ Desplazamientos: {self.desplazamientos}") 
        print(f"💰 Coste total: {self.coste_acumulado:.2f}€") 
        print(f"🗑️ Espacio desperdiciado: {self.espacio_desperdiciado} uds") 

In [32]:
import sys

In [33]:
async def ejecutar_simulacion_simple(runtime, estado):
    metricas = MonitorMetricas()
    iteracion = 0
    simulacion_activa = True
    penalizacion_t = estado.penalizacion_trafico

    while simulacion_activa and iteracion < 150:
        iteracion += 1
        metricas.tiempo_total = iteracion
        
        # Guardar posiciones antes del mensaje
        pos_antes = {v.id: v.posicion for v in estado.vehiculos}
        
        for v in estado.vehiculos:
            await runtime.send_message(estado, AgentId(v.id, "default"))
        
        await asyncio.sleep(0.05) 
        
        # Actualizar métricas si hubo movimiento
        for v in estado.vehiculos:
            if v.posicion != pos_antes[v.id]:
                # Registramos métricas
                tipo_celda = estado.mapa[v.posicion[0]][v.posicion[1]]
                metricas.registrar_paso(v, tipo_celda, penalizacion_t)
        
        imprimir_mapa_coloreado(estado)
        
        pendiente = sum(sum(c.demandas.values()) - sum(c.recibido.values()) for c in estado.clientes)
        if pendiente == 0:
            simulacion_activa = False
            print(f"✅ ¡Simulación finalizada!")
            
    # 5. Mostrar resultados finales [cite: 48]
    metricas.mostrar_resumen()

In [34]:
# 1. Entorno y Datos
runtime = SingleThreadedAgentRuntime()
estado_inicial = cargar_configuracion("escenario_logistica.json")

# 2. Registro de agentes con corrección de Lambda
for v_data in estado_inicial.vehiculos:
    # Usamos v=v_data para capturar la instancia correcta en cada iteración
    await AgenteVehiculoSimple.register(
        runtime, 
        type=v_data.id, 
        factory=lambda v=v_data: AgenteVehiculoSimple(
            id=v.id, 
            capacidad=v.capacidad_max, 
            coste_paso=v.coste_paso, 
            posicion_inicial=v.posicion,
            penalizacion_t = estado_inicial.penalizacion_trafico
        )
    )

# 3. Ejecución
runtime.start() # [cite: 544]
await ejecutar_simulacion_simple(runtime, estado_inicial)
await runtime.stop() # [cite: 545]
await runtime.close() # [cite: 549]

🚚 Vehículo V1/default se mueve a (7, 2) (Carga: 0)

--- ITERACIÓN ---
. . . . . # . . . .
. D . . . # . T T .
. . . . . . . T C .
. # # # . . . . . .
. . . . . . . . . .
. C . T . D . . . .
. . . T . . . # # .
. . V . . . . # . .
. . . . . . . . . .
. . . . . . . . . .
🚚 Vehículo V1/default se mueve a (7, 3) (Carga: 0)

--- ITERACIÓN ---
. . . . . # . . . .
. D . . . # . T T .
. . . . . . . T C .
. # # # . . . . . .
. . . . . . . . . .
. C . T . D . . . .
. . . T . . . # # .
. . . V . . . # . .
. . . . . . . . . .
. . . . . . . . . .
🚚 Vehículo V1/default se mueve a (6, 4) (Carga: 0)

--- ITERACIÓN ---
. . . . . # . . . .
. D . . . # . T T .
. . . . . . . T C .
. # # # . . . . . .
. . . . . . . . . .
. C . T . D . . . .
. . . T V . . # # .
. . . . . . . # . .
. . . . . . . . . .
. . . . . . . . . .
🚚 Vehículo V1/default se mueve a (5, 5) (Carga: 0)
📦 V1/default cargó 10 de D2. Total: 10

--- ITERACIÓN ---
. . . . . # . . . .
. D . . . # . T T .
. . . . . . . T C .
. # # # . . . . . .
.

Entorno coordinado

In [35]:
class SolicitudSuministro(BaseModel): # Enviado por Cliente -> Coordinador
    id_cliente: str
    posicion_entrega: Tuple[int, int]
    prioridad: Prioridad
    articulos: Dict[str, int] # {id_deposito: cantidad}

class CallForProposal(BaseModel):
    id_subasta: str
    id_cliente: str
    pos_cliente: Tuple[int, int]
    id_deposito: str
    pos_deposito: Tuple[int, int] # <--- Nueva inclusión obligatoria
    cantidad: int
    prioridad: Prioridad

class Adjudicacion(BaseModel):
    id_subasta: str
    aceptado: bool
    id_deposito: Optional[str] = None
    pos_deposito: Optional[Tuple[int, int]] = None # <--- Nueva inclusión
    id_cliente: Optional[str] = None
    pos_cliente: Optional[Tuple[int, int]] = None
    cantidad: Optional[int] = None

class Proposal(BaseModel):
    id_subasta: str
    id_vehiculo: str
    coste_estimado: float
    puede_realizar: bool

class ConsultaPosicion(BaseModel): # Vehículo -> Depósito
    id_vehiculo: str

class RespuestaPosicion(BaseModel): # Depósito -> Vehículo
    id_deposito: str
    posicion: Tuple[int, int]

class MensajeInicio(BaseModel):
    comando: str

class PeticionCarga(BaseModel):
    volumen_solicitado: int

class RespuestaCarga(BaseModel):
    id_deposito: str
    pares_entregados: int
    impares_entregados: int
    pares_restantes: int      # Lo que le queda al depósito
    impares_restantes: int    # Lo que le queda al depósito
    volumen_solicitado_original: int # Para que el vehículo ajuste sus reservas

In [ ]:
class AgenteCliente(RoutedAgent):
    def __init__(self, id, posicion, demandas, prioridad):
        super().__init__(id)
        self.posicion = posicion
        self.demandas = demandas
        self.prioridad = prioridad
        self.recibido = {}

    @message_handler
    async def solicitar_ayuda(self, message:MensajeInicio , ctx) -> None:
        # El cliente levanta la mano si tiene demanda pendiente
        pendiente = {d: cant - self.recibido.get(d, 0) for d, cant in self.demandas.items() if cant > self.recibido.get(d, 0)}
        if pendiente:
            await self.send_message(
                SolicitudSuministro(id_cliente=self.id.type, posicion_entrega=self.posicion, 
                                   prioridad=self.prioridad, articulos=pendiente),
                recipient=AgentId("coordinador", "default")
            )

In [37]:
import pandas as pd

class AgenteMonitor(RoutedAgent):
    def __init__(self, id):
        super().__init__(id)
        self.registro_eventos = []

    @message_handler
    async def registrar_evento(self, message: MensajeLog, ctx) -> None:
        """Guarda cada evento que ocurre en la simulación."""
        self.registro_eventos.append({
            "Iteracion": message.iteracion,
            "Agente": message.agente,
            "Accion": message.accion,
            "Detalles": message.detalles
        })

    def exportar_datos(self, nombre_archivo="metricas_simulacion.csv"):
        """Convierte el historial a un DataFrame y lo guarda."""
        if self.registro_eventos:
            df = pd.DataFrame(self.registro_eventos)
            df.to_csv(nombre_archivo, index=False)
            print(f"📊 [Monitor] Datos exportados exitosamente a {nombre_archivo}")

In [38]:
class AgenteDeposito(RoutedAgent):
    def __init__(self, id, posicion, pares, impares):
        super().__init__(id)
        self.posicion = posicion
        self.paquetes_pares = pares
        self.paquetes_impares = impares

    @message_handler
    async def procesar_peticion_carga(self, message: PeticionCarga, ctx) -> None:
        """El depósito recibe la petición, procesa su stock y envía lo que ha podido cargar."""
        vol_pendiente = message.volumen_solicitado
        
        # 1. Prioridad: Paquetes pares (ocupan 2 uds de volumen)
        pares_a_dar = min(vol_pendiente // 2, self.paquetes_pares)
        self.paquetes_pares -= pares_a_dar
        vol_pendiente -= (pares_a_dar * 2)
        
        # 2. Rellenar con paquetes impares (ocupan 1 ud)
        impares_a_dar = min(vol_pendiente, self.paquetes_impares)
        self.paquetes_impares -= impares_a_dar
        
        # 3. Enviar respuesta al vehículo con la mercancía y el stock restante
        respuesta = RespuestaCarga(
            id_deposito=self.id.type,
            pares_entregados=pares_a_dar,
            impares_entregados=impares_a_dar,
            pares_restantes=self.paquetes_pares,     # Manda sus restantes
            impares_restantes=self.paquetes_impares, # Manda sus restantes
            volumen_solicitado_original=message.volumen_solicitado
        )
        
        await self.send_message(respuesta, recipient=ctx.sender)
        

In [ ]:
class AgenteCoordinador(RoutedAgent):
    def __init__(self, id, num_clientes, num_vehiculos, depositos_info):
        super().__init__(id)
        self.num_clientes = num_clientes
        self.num_vehiculos = num_vehiculos
        self.peticiones_pendientes = []
        self.respuestas_lote = []
        self.indice_subasta_actual = 0
        self.tareas_ordenadas = []
        self.mapa_depositos = {d.id: d.posicion for d in depositos_info}
        self.fase_subasta_completa = False

    @message_handler
    async def iniciar_proceso_logistico(self, message: MensajeInicio, ctx) -> None:
        """El orquestador despierta a todos los clientes."""
        print("📢 Coordinador: Iniciando fase de descubrimiento de demanda...")
        for i in range(1, self.num_clientes + 1):
            # Asumiendo que los IDs de cliente siguen el patrón 'C1', 'C2', etc.
            await self.send_message(MensajeInicio(comando="START"), AgentId(f"C{i}", "default"))

    @message_handler
    async def recibir_solicitudes_iniciales(self, message: SolicitudSuministro, ctx) -> None:
        """Fase 1: Recopilar todas las solicitudes iniciales."""
        self.peticiones_pendientes.append(message) 

        # Una vez recibidas todas las de los clientes, ordenamos y empezamos
        if len(self.peticiones_pendientes) == self.num_clientes:
            print("📦 Coordinador: Todas las peticiones recibidas. Ordenando por prioridad...")
            self._preparar_y_ordenar_lote()
            await self.iniciar_siguiente_subasta()

    def _preparar_y_ordenar_lote(self):
        """Ordenación: 1º Prioridad (desc), 2º Cantidad (desc)."""
        for sol in self.peticiones_pendientes:
            for dep_id, cant in sol.articulos.items():
                self.tareas_ordenadas.append({
                    "id_subasta": f"sub_{sol.id_cliente}_{dep_id}",
                    "prioridad": sol.prioridad,
                    "cantidad": cant,
                    "id_cliente": sol.id_cliente,
                    "pos_cliente": sol.posicion_entrega,
                    "id_deposito": dep_id,
                    "pos_deposito": self.mapa_depositos.get(dep_id),
                })
        # Aplicamos el criterio de ordenación doble
        self.tareas_ordenadas.sort(key=lambda x: (-x["prioridad"].value, -x["cantidad"]))
    
    async def iniciar_siguiente_subasta(self):
        """Fase 2: Lanzar CFP para la tarea actual del lote."""
        if self.indice_subasta_actual < len(self.tareas_ordenadas):
            tarea = self.tareas_ordenadas[self.indice_subasta_actual]            
            self.respuestas_lote = []
            cfp = CallForProposal(
                id_subasta=tarea["id_subasta"],
                id_cliente=tarea["id_cliente"],
                pos_cliente=tarea["pos_cliente"],
                id_deposito=tarea["id_deposito"],
                pos_depositio=tarea["id_deposito"],
                cantidad=tarea["cantidad"],
                prioridad=tarea["prioridad"]
            )
            await self.publish_message(cfp, topic_id=DefaultTopicId())
        else:
            print("🏁 Lote de subastas finalizado.")
            self.fase_subasta_completa = True

    @message_handler
    async def gestionar_propuestas(self, message: Proposal, ctx) -> None:
        """Fase 3: Evaluar Propose/Refuse y adjudicar."""
        self.respuestas_lote.append(message)
        
        if len(self.respuestas_lote) == self.num_vehiculos:
            await self.seleccionar_ganador_y_continuar()
    
    async def seleccionar_ganador_y_continuar(self):
        tarea = self.tareas_ordenadas[self.indice_subasta_actual]
        validas = [r for r in self.respuestas_lote if r.puede_realizar]
        if validas:
            # 1. OPTIMIZACIÓN: Normalización dinámica (Max-Min)
            # Obtenemos los valores máximos de esta ronda exacta para normalizar entre 0 y 1
            max_coste = max((p.coste_estimado for p in validas), default=1)
            if max_coste == 0: max_coste = 1 # Evitar división por cero
            
            max_cap = max((p.capacidad_libre_tras_tarea for p in validas), default=1)
            if max_cap == 0: max_cap = 1

            def calcular_puntuacion(p):
                # Factor Coste: 0.0 es el mejor, 1.0 es el peor
                f_coste = p.coste_estimado / max_coste 
                
                # Factor Desperdicio: 0.0 es el mejor (camión lleno), 1.0 es el peor (camión vacío)
                # Esto favorece la "Consolidación": usar menos camiones pero más llenos.
                f_desperdicio = p.capacidad_libre_tras_tarea / max_cap 
                
                # Puntuación Final: Damos 85% de importancia a no hacer kilómetros extra, 
                # y un 15% de importancia a aprovechar bien el espacio del camión.
                return (0.85 * f_coste) + (0.15 * f_desperdicio)

            # Seleccionamos al que minimiza la función
            ganador = min(validas, key=calcular_puntuacion)
            
            print(f"🏆 Ganador para subasta {tarea['id_subasta']}: {ganador.id_vehiculo}")
            print(f"   (Coste marginal: {ganador.coste_estimado}, Espacio restante: {ganador.capacidad_libre_tras_tarea})")

            for resp in self.respuestas_lote:
                es_ganador = (resp.id_vehiculo == ganador.id_vehiculo)
                await self.send_message(
                    Adjudicacion(
                        id_subasta=tarea["id_subasta"], aceptado=es_ganador,
                        id_deposito=tarea["id_deposito"], 
                        pos_deposito=tarea["pos_deposito"], 
                        cantidad=tarea["cantidad"],
                        id_cliente=tarea["id_cliente"], pos_cliente=tarea["pos_cliente"]
                    ),
                    recipient=AgentId(resp.id_vehiculo, "default")
                )
        else:
            print(f"⚠️ Ningún vehículo pudo asumir la tarea {tarea['id_subasta']}")
        
        self.indice_subasta_actual += 1
        await self.iniciar_siguiente_subasta()

In [40]:
class AgenteVehiculoCoordinado(RoutedAgent):
    def __init__(self, id, capacidad, coste_paso, penalizacion_t, posicion_inicial):
        super().__init__(id)
        # Atributos físicos y de configuración
        self.capacidad_max = capacidad
        self.coste_paso = coste_paso
        self.posicion = posicion_inicial
        self.penalizacion_t = penalizacion_t
        
        # Estado de carga
        self.carga_actual = 0
        self.capacidad_reservada = 0  # Carga ganada en subasta que aún no recogemos
        self.inventario_interno = {}      
          
        # Memoria y Planificación
        self.plan_paradas_futuras = [] # Destinos lógicos: [(tipo, pos, id, cant)]
        self.plan_ruta_pasos = []      # Celdas específicas del A*
        self.mapa_memoria = None       # Copia del mapa para cálculos
        self.cache_distancias = {}

    def _simular_costo_plan(self, plan):
        """Calcula el costo total en pasos de una secuencia de paradas."""
        if not plan: 
            return 0 
        costo_total = 0
        punto_actual = self.posicion
        for _, pos_destino, _, _ in plan:
            costo_total += self.estimar_coste_A_estrella(punto_actual, pos_destino)
            punto_actual = pos_destino
        return costo_total

    def _es_plan_valido(self, plan):
        """Verifica si en algún punto de la ruta se supera la capacidad máxima."""
        carga_simulada = self.carga_actual
        for tipo, _, _, cant in plan:
            if tipo == "RECOGER":
                carga_simulada += cant
            elif tipo == "ENTREGAR":
                carga_simulada -= cant
            if carga_simulada > self.capacidad_max:
                return False
        return True

    @message_handler
    async def evaluar_cfp(self, message: CallForProposal, ctx) -> None:
        espacio_libre = self.capacidad_max - self.carga_actual - self.capacidad_reservada
        if espacio_libre < message.cantidad:
            await self.send_message(Proposal(id_subasta=message.id_subasta, id_vehiculo=self.id.type, 
                                            coste_estimado=float('inf'), capacidad_libre_tras_tarea=0, puede_realizar=False), ctx.sender)
            return

        costo_actual = self._simular_costo_plan(self.plan_paradas_futuras)
        mejor_incremento = float('inf')
        
        nueva_recogida = ("RECOGER", message.pos_deposito, message.id_deposito, message.cantidad, message.id_deposito)
        nueva_entrega = ("ENTREGAR", message.pos_cliente, message.id_cliente, message.cantidad, message.id_deposito)

        n = len(self.plan_paradas_futuras)
        for i in range(n + 1):
            for j in range(i + 1, n + 2):
                plan_test = list(self.plan_paradas_futuras)
                plan_test.insert(i, nueva_recogida)
                plan_test.insert(j, nueva_entrega)

                if self._es_plan_valido(plan_test):
                    incremento = self._simular_costo_plan(plan_test) - costo_actual
                    if incremento < mejor_incremento:
                        mejor_incremento = incremento

        if mejor_incremento != float('inf'):
            await self.send_message(Proposal(id_subasta=message.id_subasta, id_vehiculo=self.id.type,
                coste_estimado=mejor_incremento, capacidad_libre_tras_tarea=(espacio_libre - message.cantidad), puede_realizar=True), ctx.sender)

    @message_handler
    async def recibir_adjudicacion(self, message: Adjudicacion, ctx) -> None:
        """Si gana, compromete espacio y añade paradas al plan maestro."""
     
        if message.aceptado:
            self.capacidad_reservada += message.cantidad
            pos_dep = message.pos_deposito
            
            nueva_recogida = ("RECOGER", pos_dep, message.id_deposito, message.cantidad)
            nueva_entrega = ("ENTREGAR", message.pos_cliente, message.id_cliente, message.cantidad)
            
            # Repetimos la lógica de búsqueda para aplicar la mejor inserción real
            mejor_i, mejor_j = 0, 1
            min_costo = float('inf')
            
            n = len(self.plan_paradas_futuras)
            for i in range(n + 1):
                for j in range(i + 1, n + 2):
                    plan_test = list(self.plan_paradas_futuras)
                    plan_test.insert(i, nueva_recogida)
                    plan_test.insert(j, nueva_entrega)
                    if self._es_plan_valido(plan_test):
                        c = self._simular_costo_plan(plan_test)
                        if c < min_costo:
                            min_costo = c
                            mejor_i, mejor_j = i, j

            self.plan_paradas_futuras.insert(mejor_i, nueva_recogida)
            self.plan_paradas_futuras.insert(mejor_j, nueva_entrega)
            
            # Forzar recalcular ruta A* si cambiamos el primer destino
            self.plan_ruta_pasos = [] 
            print(f"🏆 {self.id.type} ganó {message.id_subasta} (Inserción en pos {mejor_i} y {mejor_j})")
    
    @message_handler
    async def actualizar_y_decidir(self, message: EstadoEntorno, ctx) -> None:
        """Motor de movimiento y lógica de entrega/carga."""
        self.mapa_memoria = message.mapa # Actualizar mapa (por si hay cambios)

        # 1. Si no hay ruta pero hay paradas, calcular A* al siguiente destino
        if not self.plan_ruta_pasos and self.plan_paradas_futuras:
            tipo, destino, _, _ = self.plan_paradas_futuras[0]
            if self.posicion == destino:
                await self.verificar_llegada_coordinada(message) # Llegada inmediata
            else:
                self.plan_ruta_pasos = self.calcular_a_estrella(self.posicion, destino, message.mapa)
        
        # 2. Movimiento físico
        if self.plan_ruta_pasos:
            self.posicion = self.plan_ruta_pasos.pop(0)
            self._sincronizar_con_entorno(message) # Crucial para el visualizador
            
            # 3. Verificar si hemos llegado al destino lógico
            if not self.plan_ruta_pasos:
                await self.verificar_llegada_coordinada(message)

    async def verificar_llegada_coordinada(self, estado):
        """Lógica de carga en depósito o entrega en cliente."""
        if not self.plan_paradas_futuras: 
            return
        tipo, pos, entidad_id, cant, id_origen = self.plan_paradas_futuras.pop(0)        
        if tipo == "RECOGER":
            await self.send_message(
                PeticionCarga(volumen_solicitado=cant),
                recipient=AgentId(id_origen, "default")
            )
        elif tipo == "ENTREGAR":
            self.carga_actual -= cant
            self.inventario_interno[entidad_id] = 0 # Asumimos entrega total de ese origen
            # Actualizar progreso del cliente en el mensaje global
            for c in estado.clientes:
                if c.id == entidad_id:
                    c.recibido[id_origen] = c.recibido.get(id_origen, 0) + cant
            print(f"🚚 {self.id.type} entregó {cant} a {entidad_id}")

    @message_handler
    async def recibir_carga(self, message: RespuestaCarga, ctx) -> None:
        """El vehículo procesa la carga física que le acaba de entregar el depósito."""
        volumen_real_recibido = (message.pares_entregados * 2) + message.impares_entregados
        
        self.carga_actual += volumen_real_recibido
        self.capacidad_reservada -= message.volumen_solicitado_original
        self.inventario_interno[message.id_deposito] = self.inventario_interno.get(message.id_deposito, 0) + volumen_real_recibido
        
        print(f"📦 {self.id.type} cargó de {message.id_deposito}: {message.pares_entregados} pares y {message.impares_entregados} impares.")
        print(f"   -> Info del {message.id_deposito}: Me quedan {message.pares_restantes} pares y {message.impares_restantes} impares en el almacén.")

        
    # --- Métodos Auxiliares de Navegación ---
    def estimar_coste_A_estrella(self, inicio, fin):
        if inicio == fin :
            return 0
        ruta = self.calcular_a_estrella(inicio, fin, self.mapa_memoria)
        
        # 1. Comprobamos si ya conocemos esta distancia
        tupla_ruta = (inicio, fin)
        if tupla_ruta in self.cache_distancias:
            return self.cache_distancias[tupla_ruta]

        # 2. Si no la conocemos, ejecutamos el A* pesado
        ruta = self.calcular_a_estrella(inicio, fin, self.mapa_memoria)
        coste = 0
        for r, c in ruta:
            coste += self.coste_paso
            if self.mapa_memoria[r][c] == 'T': 
                coste += self.penalizacion_t
                
        # 3. Guardamos el resultado en caché para el futuro
        self.cache_distancias[tupla_ruta] = coste
        
        # Opcional: Asumir que el coste de vuelta es el mismo (si no hay calles de un sentido)
        self.cache_distancias[(fin, inicio)] = coste 
        
        return coste

    def calcular_a_estrella(self, inicio, fin, mapa):
        if not mapa or inicio == fin: return []
        direcciones = [(0, 1), (0, -1), (1, 0), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1)]
        open_list = [(0, inicio)]
        camino_previo = {}
        coste_acumulado = {inicio: 0}
        while open_list:
            _, actual = heapq.heappop(open_list)
            if actual == fin:
                camino = []
                while actual in camino_previo:
                    camino.append(actual)
                    actual = camino_previo[actual]
                return camino[::-1]
            for dx, dy in direcciones:
                vecino = (actual[0] + dx, actual[1] + dy)
                if 0 <= vecino[0] < len(mapa) and 0 <= vecino[1] < len(mapa[0]) and mapa[vecino[0]][vecino[1]] != '#':
                    coste = self.coste_paso + (self.penalizacion_t if mapa[vecino[0]][vecino[1]] == 'T' else 0)
                    nuevo_coste = coste_acumulado[actual] + coste
                    if vecino not in coste_acumulado or nuevo_coste < coste_acumulado[vecino]:
                        coste_acumulado[vecino] = nuevo_coste
                        prioridad = nuevo_coste + max(abs(vecino[0]-fin[0]), abs(vecino[1]-fin[1]))
                        heapq.heappush(open_list, (prioridad, vecino))
                        camino_previo[vecino] = actual
        return []
        
    def _sincronizar_con_entorno(self, message):
        for v in message.vehiculos:
            if v.id == self.id.type:
                v.posicion = self.posicion
                v.carga_actual = self.carga_actual

In [41]:
async def inicializar_escenario_coordinado(runtime, estado):

    # await AgenteMonitor.register(
    #     runtime, type="monitor_analitica",
    #     factory=lambda: AgenteMonitor("monitor_analitica")
    # )
    # 1. Registrar Agente Coordinador
    await AgenteCoordinador.register(
        runtime, 
        type="coordinador", 
        factory=lambda: AgenteCoordinador(
            id="coordinador", 
            num_clientes=len(estado.clientes), 
            num_vehiculos=len(estado.vehiculos),
            depositos_info=estado.depositos
        )
    )

    # 2. Registrar Agentes Cliente
    for c in estado.clientes:
        await AgenteCliente.register(
            runtime,
            type=c.id,
            factory=lambda c=c: AgenteCliente(c.id, c.posicion, c.demandas, c.prioridad)
        )

    # 3. Registrar Agentes Depósito
    for d in estado.depositos:
        await AgenteDeposito.register(
            runtime,
            type=d.id,
            factory=lambda d=d: AgenteDeposito(d.id, d.posicion, d.paquetes_pares, d.paquetes_impares)
        )

    # 4. Registrar Agentes Vehículo Coordinado
    for v in estado.vehiculos:
        await AgenteVehiculoCoordinado.register(
            runtime,
            type=v.id,
            factory=lambda v=v: AgenteVehiculoCoordinado(
                v.id, v.capacidad_max, v.coste_paso, estado.penalizacion_trafico, v.posicion
            )
        )
        await runtime.add_subscription(TypePrefixSubscription(v.id, DefaultTopicId()))

In [ ]:
async def ejecutar_simulacion_coordinada(runtime, estado):
    metricas = MonitorMetricas()
    iteracion = 0
    
    print("\n" + "="*40)
    print("🚀 INICIANDO ESCENARIO COORDINADO PROFESIONAL")
    print("="*40)

    # --- FASE 1: HANDSHAKE Y PLANIFICACIÓN ---
    # 1. El main dispara al Coordinador
    await runtime.send_message(MensajeInicio(comando="BOOT"), AgentId("coordinador", "default"))
    
    # 2. Esperamos a que el Coordinador termine las subastas (polling simple)
    coordinador = await runtime.try_get_underlying_agent_instance(AgentId("coordinador", "default"))
    while not coordinador.fase_subasta_completada:
        await asyncio.sleep(0.1) # Breve espera para que los agentes conversen
    
    print("✅ Planificación completada. Iniciando ciclo de movimiento...")

    # --- FASE 2: CICLO DE EJECUCIÓN (TICKS) ---
    while iteracion < 150:
        iteracion += 1
        estado.tick_actual = iteracion # Sincronizamos el tiempo
        metricas.tiempo_total = iteracion
        
        pos_antes = {v.id: v.posicion for v in estado.vehiculos}

        # Los vehículos deciden y se mueven basándose en el estado actualizado
        for v in estado.vehiculos:
            await runtime.send_message(estado, AgentId(v.id, "default"))
        
        await asyncio.sleep(0.05) 

        # Registro de métricas y Logs al Monitor
        for v in estado.vehiculos:
            if v.posicion != pos_antes[v.id]:
                r, c = v.posicion
                tipo_celda = estado.mapa[r][c]
                metricas.registrar_paso(v, tipo_celda, estado.penalizacion_trafico)
                
                # Llamada al Monitor Analítico
                await runtime.send_message(
                    MensajeLog(
                        iteracion=iteracion, agente=v.id, accion="MOVIMIENTO",
                        detalles=f"Moviéndose a {v.posicion}. Carga: {v.carga_actual}"
                    ),
                    AgentId("monitor_analitica", "default")
                )

        imprimir_mapa_coloreado(estado)

        # Condición de victoria
        pendiente = sum(sum(c.demandas.values()) - sum(c.recibido.values()) for c in estado.clientes)
        if pendiente == 0 and all(v.carga_actual == 0 for v in estado.vehiculos):
            print(f"\n✅ ¡Objetivo logístico cumplido en {iteracion} iteraciones!")
            break
            
    # Exportar datos finales
    monitor = await runtime.try_get_underlying_agent_instance(AgentId("monitor_analitica", "default"))
    monitor.exportar_datos()
    metricas.mostrar_resumen()

In [43]:
# Preparar entorno
runtime_coord = SingleThreadedAgentRuntime()
estado_coord = cargar_configuracion("escenario_logistica.json")

# Iniciar
await inicializar_escenario_coordinado(runtime_coord, estado_coord)
runtime_coord.start()

# Correr simulación
await ejecutar_simulacion_coordinada(runtime_coord, estado_coord)

# Limpieza
await runtime_coord.stop()


🚀 INICIANDO ESCENARIO COORDINADO
📦 Coordinador: Todas las peticiones recibidas. Ordenando por prioridad...


ValidationError: 1 validation error for CallForProposal
pos_deposito
  Field required [type=missing, input_value={'id_subasta': 'sub_C1_D1...d': <Prioridad.ALTA: 3>}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing